In [1]:
import bs4
import httpx

content_base_url = "https://www.gov.uk/guidance/content-design"


def extract_guidance_links(html_content: bs4.BeautifulSoup):
    results = html_content.select("#manuals-frontend a.govuk-link")

    return [{"link": link.get("href")} for link in results]


async def get_guidance():
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{content_base_url}")

        soup = bs4.BeautifulSoup(response.text, "lxml")

        return extract_guidance_links(soup)

guidance_pages = await get_guidance()

In [2]:
import asyncio
import re

from markdownify import markdownify as md

content_api_base_url = "https://www.gov.uk/api/content"

PAGES_TO_SPLIT = {
    "content-types": 2,
}


async def get_section_content(page):
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{content_api_base_url}{page['link']}")

        json = response.json()

        html = json["details"]["body"]

        soup = bs4.BeautifulSoup(html, "lxml")

        return {
            "title": json["title"],
            "description": json["description"],
            "content": md(str(soup), heading_style="ATX")
        }


batch_size = 10

batches = [guidance_pages[i : i + batch_size] for i in range(0, len(guidance_pages), batch_size)]

for i in range(len(batches)):
    batch = batches[i]

    contents = await asyncio.gather(*[get_section_content(page) for page in batch])

    for page, content in zip(batch, contents, strict=True):
        page["title"] = content["title"]
        page["description"] = content["description"]
        page["content"] = content["content"]

    print(f"Completed batch {i + 1} out of {len(batches)}")

    await asyncio.sleep(2)


def split_by_heading(page, level):
    slug = page["link"].split("/")[-1]
    hashes = "#" * level
    sections = []
    for part in re.split(rf"(?=^{hashes} )", page["content"], flags=re.MULTILINE):
        part = part.strip()
        if not part:
            continue
        first_line = part.split("\n")[0]
        title = first_line[level + 1:].strip() if first_line.startswith(f"{hashes} ") else "Overview"
        section_slug = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")
        sections.append({
            "link": f"{page['link']}/{section_slug}",
            "title": title,
            "description": page["description"],
            "content": part,
            "file": f"{slug}/{section_slug}.md",
        })
    return sections


for page in [p for p in guidance_pages if p["link"].split("/")[-1] in PAGES_TO_SPLIT]:
    level = PAGES_TO_SPLIT[page["link"].split("/")[-1]]
    sections = split_by_heading(page, level)
    guidance_pages.remove(page)
    guidance_pages.extend(sections)
    print(f"Split '{page['title']}' into {len(sections)} sections")

Completed batch 1 out of 3
Completed batch 2 out of 3
Completed batch 3 out of 3
Split 'Content types' into 57 sections


In [3]:
from pathlib import Path

import aiofiles

output_dir = Path("outputs/content-guidance")
output_dir.mkdir(parents=True, exist_ok=True)


async def save_page(page):
    filepath = output_dir / page.get("file", f"{page['link'].split('/')[-1]}.md")
    filepath.parent.mkdir(parents=True, exist_ok=True)

    frontmatter = f"""---
title: {page['title']}
description: {page['description']}
---

"""

    async with aiofiles.open(filepath, "w", encoding="utf-8") as f:
        await f.write(frontmatter + page["content"])


await asyncio.gather(*[save_page(page) for page in guidance_pages])

print(f"\nAll {len(guidance_pages)} markdown files saved to {output_dir.absolute()}")


All 79 markdown files saved to /home/shaun/repos/ai/ace-pod-2-patterns/ai-uc-content-swarm/ai-uc-content-swarm/core/notebooks/outputs/content-guidance


In [5]:
import json

index = [
    {
        "id": page["link"].split("/")[-1],
        "title": page["title"],
        "description": page["description"],
        "file": page.get("file", f"{page['link'].split('/')[-1]}.md"),
    }
    for page in guidance_pages
]

async with aiofiles.open(output_dir / "index.json", "w", encoding="utf-8") as f:
    await f.write(json.dumps(index))

print(f"Index saved to {(output_dir / 'index.json').absolute()}")

Index saved to /home/shaun/repos/ai/ace-pod-2-patterns/ai-uc-content-swarm/ai-uc-content-swarm/core/notebooks/outputs/content-guidance/index.json
